# 05 · Autocodificador variacional condicional (CVAE)

**Taller B5-T1 · Generación de datos financieros sinteticos**

> **Notebook pendiente de asignar.** Pietro y Alonso deben repartirse entre
> ellos este notebook, el del otro generador y el de la comparativa final.
> Conviene decidirlo antes de empezar para no duplicar trabajo.
>
> Los bloques marcados como **PENDIENTE** son los que hay que completar. El
> resto, incluida la carga de datos y el guardado de resultados, ya esta
> resuelto y no conviene modificarlo: es lo que garantiza que los resultados de
> los cuatro generadores sean comparables entre si.
>
> Antes de empezar, leer `docs/GUIA_EQUIPO.md` y usar
> `04_generador_cgan.ipynb` como referencia de estructura y de estilo.

## Preparación del entorno

Se fija el backend de cómputo, se añade el código común del proyecto a la ruta
de importación y se aplica el estilo gráfico compartido. El bloque funciona sin
cambios tanto en una instalación local como en Colab.

In [ ]:
import os
import sys

# Keras 3 se ejecuta sobre PyTorch: la versión de Python empleada no dispone de
# TensorFlow y la API de capas y modelos es idéntica en ambos backends.
os.environ["KERAS_BACKEND"] = "torch"

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    !pip install -q yfinance keras torch
    # En Colab se asume que el repositorio está clonado en el directorio actual.
    RAIZ = "/content/B5-T1"
else:
    RAIZ = os.path.dirname(os.getcwd())

if os.path.join(RAIZ, "src") not in sys.path:
    sys.path.insert(0, os.path.join(RAIZ, "src"))

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from miax_b5t1 import config, datos, experimento, graficos, modelo

graficos.aplicar_estilo()
config.asegurar_directorios()

print(f"Keras {keras.__version__} sobre backend {keras.backend.backend()}")
print(f"Raíz del proyecto: {RAIZ}")

## 1. Datos de partida

Se carga el dataset y se extrae el presupuesto de datos reales fijado en el
notebook 02.

**El generador se entrena únicamente con `X_real` e `y_real`.** No debe emplearse
el conjunto de entrenamiento completo, ni validación, ni test. Si el generador
accede a datos que el clasificador no tiene, la comparación pierde validez.

In [ ]:
d = datos.cargar_dataset()

X_real, y_real = datos.submuestra_real(
    d["X_train"], d["y_train"], config.N_REALES, config.SEMILLA)

X_val, y_val = d["X_val"], d["y_val"]
X_test, y_test = d["X_test"], d["y_test"]

n_pasos, n_activos = X_real.shape[1], X_real.shape[2]
DIMENSION = n_pasos * n_activos
X_plano = X_real.reshape(len(X_real), DIMENSION)

print(f"Presupuesto real: {X_real.shape}")
print(f"Positivas: {int(y_real.sum())} ({y_real.mean():.1%})")
print(f"Dimensión de la ventana aplanada: {DIMENSION}")

## 2. Fundamento del modelo

**PENDIENTE:** redactar dos o tres parrafos explicando el modelo. Conviene cubrir:

- Que el codificador proyecta cada ventana sobre una distribución en un espacio
  latente de dimensión reducida, y que el decodificador reconstruye la ventana a
  partir de una muestra de esa distribución.
- Que la función de pérdida combina un termino de reconstrucción con la
  divergencia de Kullback-Leibler, y para que sirve cada uno: el primero fuerza a
  que la ventana reconstruida se parezca a la original, el segundo a que el
  espacio latente sea continuo y muestreable.
- Que hace falta el truco de la reparametrización para poder derivar a través de
  la operación de muestreo.
- Por que se condiciona por la etiqueta: al incorporarla como entrada a ambas
  redes, se puede pedir al decodificador muestras de la clase positiva de forma
  explicita, que es lo que interesa dada la escasez de episodios de estrés.
- Que cabe esperar del modelo en este problema. Un CVAE tiende a producir
  muestras suavizadas, porque el termino de reconstrucción penaliza el error
  cuadratico y el optimo ante la incertidumbre es promediar. Conviene anticiparlo
  y comprobarlo despues en la validación cualitativa.

## 3. Arquitectura

**PENDIENTE:** definir codificador y decodificador.

Orientaciones:

- Trabajar sobre la ventana aplanada, de dimensión `DIMENSION`, con capas densas,
  igual que hace el cGAN del notebook 04.
- Codificador: entrada la ventana aplanada concatenada con la etiqueta; salida
  `z_media` y `z_log_var`, cada una de dimensión `DIM_LATENTE`. Un valor de
  partida razonable para `DIM_LATENTE` esta entre 16 y 64.
- Decodificador: entrada el vector latente concatenado con la etiqueta; salida la
  ventana aplanada con activación `tanh`, ya que los datos estan escalados al
  intervalo `[-1, 1]`.
- El muestreo latente se implementa con una capa `Lambda` o con una subclase de
  `keras.layers.Layer`.

Aviso sobre el entorno: se emplea Keras 3 sobre PyTorch. Las funciones de
`keras.ops` funcionan en cualquier backend, pero `tf.random.normal` no esta
disponible. Para generar el ruido de la reparametrización debe usarse
`keras.random.normal`.

Al terminar, llamar a `summary()` en ambas redes.

In [ ]:
from keras import layers, Model
import keras.ops as ops

DIM_LATENTE = 32   # ajustar si procede

# PENDIENTE: definir el codificador, el decodificador y el modelo CVAE completo.
# Recordar llamar a summary() sobre ambas redes.

## 4. Entrenamiento

**PENDIENTE:** entrenar el modelo registrando la pérdida epoch a epoch.

Guardar el historial en una variable llamada `historial_perdida`, que es la que
recoge el bloque de guardado del final. Si se registran por separado el termino
de reconstrucción y el de divergencia, conviene representarlos también: su
evolución relativa explica el comportamiento del modelo mejor que la pérdida
total.

Un punto de partida razonable son 200 epochs con lotes de 64 muestras. Conviene
vigilar que la divergencia no caiga a cero, sintoma de que el decodificador esta
ignorando el espacio latente.

In [ ]:
# PENDIENTE: entrenamiento del CVAE.
# historial_perdida = ...

## 5. Curva de convergencia y generación

La curva de pérdida es obligatoria según el enunciado. Despues se generan las
muestras sinteticas muestreando el espacio latente y decodificando.

In [ ]:
# PENDIENTE: representar la curva de pérdida.
# graficos.curva_perdida(historial, "Convergencia del CVAE", "05_perdida_cvae")

Se generan al menos tantas muestras como exige el ratio máximo del barrido. Las
etiquetas se sortean con la misma proporción de positivos que los datos reales, y
se pasan al decodificador junto con el vector latente.

In [ ]:
N_SINTETICOS = int(config.N_REALES * max(config.RATIOS_SINTETICOS))

# PENDIENTE: generar X_synth con forma (N_SINTETICOS, n_pasos, n_activos)
#            y y_synth con forma (N_SINTETICOS,).
#
# X_synth = ...
# y_synth = ...

# print(f"Muestras sinteticas: {X_synth.shape}")
# print(f"Positivas: {y_synth.mean():.1%}")
# print(f"Rango: [{X_synth.min():.3f}, {X_synth.max():.3f}]")

## 6. Validación cualitativa

Antes de medir el efecto sobre el clasificador conviene comprobar si las muestras
generadas se parecen a ventanas de mercado reales. El análisis exploratorio
identifico tres propiedades que un generador debería reproducir: colas más
pesadas que las de una normal, agrupamiento de la volatilidad dentro de la
ventana y correlación positiva entre activos.

Conviene comprobar también que las trayectorias no sean planas ni esten saturadas
en los extremos del intervalo, dos sintomas habituales de un generador que no ha
convergido.

In [ ]:
graficos.comparar_trayectorias(
    X_real, X_synth, "Ventanas reales frente a sinteticas", "05_trayectorias")
plt.show()

In [ ]:
graficos.comparar_distribuciones(
    X_real, X_synth, "Propiedades estadisticas de las muestras generadas",
    "05_distribuciones")
plt.show()

La última comprobación mide cuanta novedad aportan las muestras. Un generador que
se limite a reproducir el conjunto de entrenamiento producira muestras muy
próximas a las reales, y en ese caso no puede aportar información que el
clasificador no tuviera ya.

In [ ]:
fig, ax, resumen_novedad = graficos.novedad_vecino_mas_cercano(
    X_real, X_synth, "Novedad de las muestras generadas", "05_novedad")
plt.show()

for clave, valor in resumen_novedad.items():
    print(f"{clave}: {valor:.4f}")

**PENDIENTE:** comentar aquí que reproduce bien el generador y que no, apoyandose
en las tres figuras anteriores. Interesa el detalle concreto: si la correlación
entre activos se conserva, si las colas son comparables a las reales, si las
muestras aportan configuraciones nuevas o son variantes de las existentes.

## 7. Evaluación

Se ejecuta el protocolo común de evaluación. La función `barrido_ratios` entrena
el clasificador con el presupuesto real más cantidades crecientes de datos
sinteticos y repite cada configuración con varias semillas.

No conviene sustituir esta llamada por un bucle propio: es lo que asegura que los
resultados de los cuatro generadores se hayan obtenido en condiciones identicas y
puedan compararse en el notebook final.

In [ ]:
resultados, historias = experimento.barrido_ratios(
    "cvae", X_real, y_real, X_synth, y_synth, X_val, y_val, X_test, y_test)

experimento.guardar_sinteticos("cvae", X_synth, y_synth,
                               pérdidas={"total": historial_perdida})
ruta = experimento.guardar_resultados(resultados, "cvae")
print(f"\nResultados guardados en {ruta}")

resumen = experimento.resumir(resultados)
display(resumen.round(4))

In [ ]:
graficos.curva_ratios(
    resumen, titulo="Efecto del generador sobre el test",
    nombre_fichero="cvae_curva_ratios")
plt.show()

base = resumen.loc[resumen["ratio"] == 0.0, "pr_auc_media"].iloc[0]
mejor = resumen.loc[resumen["pr_auc_media"].idxmax()]
print(f"PR-AUC sin sinteticos: {base:.4f}")
print(f"Mejor PR-AUC:          {mejor['pr_auc_media']:.4f} con ratio {mejor['ratio']:g}")

Las curvas de pérdida de cada entrenamiento del clasificador documentan la
convergencia exigida por el enunciado.

In [ ]:
ratios_mostrados = sorted({r for r, _ in historias})
fig, axes = plt.subplots(1, len(ratios_mostrados),
                         figsize=(3.0 * len(ratios_mostrados), 3.2), sharey=True)
axes = np.atleast_1d(axes)

for ax, ratio in zip(axes, ratios_mostrados):
    for (r, semilla), historia in historias.items():
        if r == ratio:
            ax.plot(historia["val_loss"], linewidth=1.2, alpha=0.85)
    ax.set_title(f"ratio {ratio:g}")
    ax.set_xlabel("epoch")

axes[0].set_ylabel("pérdida en validación")
fig.suptitle("Convergencia del clasificador por cantidad de sinteticos",
             fontweight="bold")
fig.tight_layout()
graficos.guardar(fig, "cvae_perdidas_clasificador")
plt.show()

## Conclusiones del notebook

**PENDIENTE:** cuatro o cinco puntos que respondan a:

1. Que propiedades de los datos reproduce el generador y cuales no.
2. Si las muestras aportan configuraciones nuevas o son variantes de las
   existentes.
3. Que efecto tiene sobre el clasificador al aumentar la proporción de
   sinteticos, en validación y en test.
4. Como se compara con el generador por ruido del notebook 03, que marca el nivel
   mínimo exigible.
5. Que limitación del modelo explica el resultado obtenido.

Los resultados desfavorables se documentan igual que los favorables. Si el
generador no mejora al clasificador, lo que se valora es explicar por que.